# PaddyGuard-TN: train the disease classifier

Thin notebook: it only sets up paths and calls the scripts. Attach two datasets (Add Input): `paddyguard-code` and `paddyguard-processed`. Settings: Accelerator = GPU, Internet = On (the pretrained weights are downloaded).

In [ ]:
import glob
import os

import torch


def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input: attach the dataset with Add Input"
    return hits[0]

code_dir = os.path.dirname(find("pyproject.toml"))
data_dir = os.path.dirname(os.path.dirname(find("manifests/manifest.csv")))
print("code:", code_dir, "\ndata:", data_dir)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn the accelerator on)")

In [ ]:
!rm -rf /kaggle/working/paddyguard && cp -r {code_dir} /kaggle/working/paddyguard
%cd /kaggle/working/paddyguard
!pip install -q timm "albumentations>=2.0" imagehash
os.environ["PYTHONPATH"] = "/kaggle/working/paddyguard/src"  # no `pip install -e .` needed
os.environ["PADDYGUARD_DATA"] = data_dir
os.environ["PADDYGUARD_OUT"] = "/kaggle/working"
!python -c "import timm, albumentations, paddyguard; print('timm', timm.__version__, '| albumentations', albumentations.__version__, '| paddyguard OK')"


## 1. Quick check (about a minute): 200 images, 1 epoch

In [ ]:
!python -m paddyguard.classify.train --config configs/cls_effb0.yaml --seed 0 --debug

## 2. Real run (EfficientNet-B0, 15 epochs). Change `--seed` to 1 and 2 for the other seeds.

In [ ]:
!python -m paddyguard.classify.train --config configs/cls_effb0.yaml --seed 0

## 3. Results

Everything is in `/kaggle/working/runs/<run>/`: `metrics.json`, `preds_S_test.parquet`, `ckpt.pt`. Download the run folder from the Output tab.

In [ ]:
import json
import pathlib

run = max(pathlib.Path("/kaggle/working/runs").glob("cls_effb0-*"))
m = json.loads((run / "metrics.json").read_text())
print(run.name, "| best epoch", m["best_epoch"])
print("test macro-F1", round(m["test"]["macro_f1"], 4), "| balanced acc", round(m["test"]["balanced_accuracy"], 4))
print("acceptance:", m["acceptance"])

## If the session stops (resume)

Re-run the setup cells, then use the run folder name from `/kaggle/working/runs`:

In [ ]:
# !python -m paddyguard.classify.train --config configs/cls_effb0.yaml --resume /kaggle/working/runs/<run folder>